In [ ]:
from fpl import db
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

from fpl.config import RECENCY_DECAY_RATE, STATS_LOOKBACK_DAYS

shots_df = db.read_table("understat_shots")
fixtures_df = db.read_table("understat_fixtures")
fixtures_df["date"] = pd.to_datetime(fixtures_df["date"])

lookback_start = datetime.now() - timedelta(days=STATS_LOOKBACK_DAYS)
fixtures_df = fixtures_df[fixtures_df["date"] >= lookback_start]

shots_df = shots_df.merge(fixtures_df[["match_id", "date"]], on="match_id")
shots_df["days_ago"] = (datetime.now() - shots_df["date"]).dt.days
# shots_df["weight"] = np.exp(-RECENCY_DECAY_RATE * shots_df["days_ago"])
shots_df["weight"] = np.exp(-0.00001 * shots_df["days_ago"])
shots_df["mins_passed_weighted"] = shots_df["mins_passed"] * shots_df["weight"]
shots_df["xg_weighted"] = shots_df["xg"] * shots_df["weight"]


In [ ]:
shots_df[shots_df['match_id'] == 31194]

In [ ]:
shots_df.columns

In [ ]:
shots_df[shots_df['match_id'] == 31194][['h_team', 'a_team', 'h_score', 'a_score', 'h_gamestate', 'a_gamestate', 'minute', 'mins_passed', 'result', 'xg', 'h_a', 'action_team', 'action_team_gamestate', 'opp_team', 'opp_team_gamestate']]

In [ ]:
# from fpl.scrape.understat import _get_json, _process_shots, _session

# data = _get_json(_session(), f"getMatchData/{31194}")
# shots = [shot for side in data["shots"].values() for shot in side]

# _process_shots(shots, 31194)[['h_team', 'a_team', 'h_score', 'a_score', 'h_gamestate', 'a_gamestate', 'minute', 'mins_passed', 'result', 'xg', 'h_a', 'action_team', 'action_team_gamestate', 'opp_team', 'opp_team_gamestate']]

In [ ]:
h_shots_df = shots_df[shots_df["h_a"] == "h"]

print(h_shots_df['xg'].sum() / h_shots_df['mins_passed'].sum())
print(h_shots_df['xg'].sum() / h_shots_df['match_id'].unique().size / 90)

print(h_shots_df['match_id'].unique().size * 90)

print(shots_df['mins_passed'].sum()/2)

print(shots_df['mins_passed'].sum()/2 / shots_df['match_id'].unique().size)

print(h_shots_df['xg'].sum() / h_shots_df['match_id'].unique().size)

In [ ]:
team_col = "action_team"
gamestate_col = "action_team_gamestate"
metric = "xg"

grouped = shots_df.pivot_table(
        values=["mins_passed_weighted", "xg_weighted", "weight"],
        index=[team_col, "h_a", gamestate_col],
        aggfunc="sum",
    )
grouped[f"{metric}_min"] = grouped["xg_weighted"] / grouped["mins_passed_weighted"]
grouped = grouped.rename_axis(index={team_col: "team", gamestate_col: "gamestate"}).reset_index()

grouped

In [ ]:
# Clear understat data from the database to re-populate
# from fpl.db import get_connection

# with get_connection() as conn:
#     cursor = conn.cursor()
#     cursor.execute("DELETE FROM understat_player_stats")
#     cursor.execute("DELETE FROM understat_rosters")
#     cursor.execute("DELETE FROM understat_shots")
#     cursor.execute("DELETE FROM understat_fixtures")
    
    
    
#     conn.commit()

In [ ]:
from fpl.scrape.understat import scrape_new_fixtures

scrape_new_fixtures(2025)
scrape_new_fixtures(2026)